# Market Access in Interwar Poland

This notebook keeps only the components needed for the final empirical results.

What is moved out to `market_access_helpers.py`:
- IO and parsing,
- scenario/year discovery,
- population preprocessing,
- export and plotting helpers.

What remains in this notebook:
- core economic formulas,
- market-access assembly logic,
- scenario execution and validation checks.


## Market Access Formula Used In This Notebook

For each origin district $i$, market access is computed as:

$MA_i = MA_i^{domestic} + MA_i^{foreign}$

Domestic component:

$MA_i^{domestic} = \sum_{j \neq i} Mass_j \cdot \exp\left(\beta_{dom}\ln d_{ij} + \gamma_t \cdot PartBorder_{ij}\right)$

where:
- $Mass_j$ is district mass (rural + city population),
- $d_{ij}$ is population-weighted district-to-district distance,
- $\beta_{dom}=-2.6705$,
- $\gamma_t$ is the year-specific partition-border coefficient from `partition_coefficients.csv`,
- $PartBorder_{ij}=1$ if districts $i$ and $j$ belonged to different historic partitions, else $0$.

Foreign component:

$MA_i^{foreign} = \sum_{r \in ForeignRegions} GDP_r \cdot \exp\left(\beta_{for}\ln D_{ir}\right)$

where:
- $GDP_r$ comes from `foreign_region_gdp.csv`,
- $D_{ir}$ is district-to-foreign-region distance via mapped border crossings,
- $\beta_{for}=-0.5684$.

Distance units:
- `fixed14`: distances are in minutes (`time_min`).
- `baseline`: distances come from horse+rail km matrices; if `COMPUTE_IN_MILES=True`, they are converted to miles before MA computation.


## 1) Setup and constants

We compute annual market access for years 1924-1938, for two scenarios:
- `baseline` (distance from horse+rail km matrices),
- `fixed14` (distance from fixed14 time matrices).

Estimated parameters used here:
- domestic distance coefficient: `-2.6705`,
- foreign distance coefficient: `-0.5684`,
- annual partition coefficient from `partition_coefficients.csv`.


In [1]:
import os
import sys
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd

os.chdir("../../..")
sys.path.append(str(Path.cwd() / "examples" / "interwar_poland"))
sys.path.append(str(Path.cwd() / "examples" / "interwar_poland" / "market_access"))

from global_definitions import adm_history_plotter, d_city_mapping
from market_access_helpers import (
    border_id_norm,
    build_border_connections,
    build_foreign_gdp_long,
    build_district_gdp_long,
    build_point_population,
    collect_target_pair_distances,
    collect_foreign_city_route_distances,
    build_population_inputs,
    list_available_years,
    load_distance_matrix_for_scenario,
    load_partition_coefficients,
    load_partition_dummies,
    match_border_connections_to_matrix_borders,
    normalize_text,
    save_annual_maps,
    save_change_map_1938_vs_1924,
    save_tables,
)

BASE_DIR = Path("examples/interwar_poland/market_access")
DATA_DIR = BASE_DIR / "data"
DIST_DIR = DATA_DIR / "distances"
OUT_DIR = BASE_DIR / "outputs" / "market_access"
PLOTS_DIR = BASE_DIR / "plots"

OUT_DIR.mkdir(parents=True, exist_ok=True)
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

DISTRICTS_GEOJSON = DATA_DIR / "districts_1934_10_1.geojson"
CITY_POP_CSV = DATA_DIR / "city_population.csv"
RURAL_POP_CSV = DATA_DIR / "rural_population.csv"
PARTITION_DUMMIES_CSV = DATA_DIR / "partition_dummies.csv"
PARTITION_COEFF_CSV = DATA_DIR / "partition_coefficients.csv"
FOREIGN_GDP_CSV = DATA_DIR / "foreign_region_gdp.csv"
DISTRICT_GDP_CSV = DATA_DIR / "district_gdp.csv"
BORDER_CONN_CSV = DATA_DIR / "border_crossing_IIRP_connections.csv"
CITY_COORDS_GEOJSON = Path("data/adm_histories/interwar_poland/cities_coords/cities_coords.geojson")

YEARS_TARGET = list(range(1924, 1939))
SCENARIOS = ["baseline", "fixed14"]
ADM_STATE_DATE = datetime(1934, 10, 1)

COEFF_DISTANCE_DOMESTIC = -2.6705
COEFF_DISTANCE_FOREIGN = -0.5684
EPS_DISTANCE = 1e-9

# Unit switch: convert all km-based distances to miles when True.
COMPUTE_IN_MILES = True
KM_TO_MILES = 0.621371

# For fixed14 we need an external connector speed in matching units.
FIXED14_EXTERNAL_SPEED_KMPH = 14.0
FIXED14_EXTERNAL_SPEED_MPH = FIXED14_EXTERNAL_SPEED_KMPH * KM_TO_MILES


Loading changes list...
✅ Loaded 309 validated changes in 0.38 seconds.
Loading initial state...
✅ Loaded initial state.
Loading initial district registry...
✅ Loaded 292 validated districts in 0.18 seconds. Set their initial state timespans to (1921-02-19, 1939-09-01).
Loading initial region registry...
✅ Loaded 19 validated regions in 0.03 seconds. Set their initial state timespans to (1921-02-19, 1939-09-01)
Creating administrative history (sequentially applying changes)...
✅ Successfully applied all changes in 22.30 seconds. Administrative history database created.
Loading territories...
Loaded: districts_1922_generalized_dp_800m.shp (271 rows)
Loaded: districts_1929_generalized_dp_800m.shp (281 rows)
Loaded: districts_1931_generalized_dp_800m.shp (283 rows)
Loaded: districts_1934_generalized_dp_800m.shp (264 rows)
Loaded: districts_1939_generalized_dp_800m.shp (265 rows)
✅ Successfully loaded all territories in 2.16 seconds.
Deducing all possible dist territories on the basis of t

## 2) Load static datasets

`District` is the canonical county identifier used throughout joins.


In [2]:
partition_coeff = load_partition_coefficients(PARTITION_COEFF_CSV)
partition_df = load_partition_dummies(PARTITION_DUMMIES_CSV)
foreign_gdp_long = build_foreign_gdp_long(FOREIGN_GDP_CSV, YEARS_TARGET)
district_gdp_long = build_district_gdp_long(DISTRICT_GDP_CSV, YEARS_TARGET)
border_conn = build_border_connections(BORDER_CONN_CSV)

districts_gdf, district_list, city_keep, rural_keep = build_population_inputs(
    districts_geojson=DISTRICTS_GEOJSON,
    city_coords_geojson=CITY_COORDS_GEOJSON,
    city_pop_csv=CITY_POP_CSV,
    rural_pop_csv=RURAL_POP_CSV,
    years_target=YEARS_TARGET,
)

print("Districts:", len(district_list))
print("Partition coeff years:", min(partition_coeff), "-", max(partition_coeff))
print("Scenarios:", SCENARIOS)
print("District GDP rows:", len(district_gdp_long))

# Self-distance from district area (equivalent-circle approach): d_ii = (2/3)*sqrt(A/pi)
# Area is computed in EPSG:3035 and converted to km^2.
_d_area = districts_gdf[["District", "geometry"]].to_crs(epsg=3035).copy()
_d_area["area_km2"] = _d_area.geometry.area / 1_000_000.0
_d_area["self_distance_km"] = (2.0 / 3.0) * np.sqrt(_d_area["area_km2"] / np.pi)
district_self_distance = _d_area[["District", "self_distance_km"]].copy()

print("District self-distance rows:", len(district_self_distance))


Districts: 247
Partition coeff years: 1924 - 1938
Scenarios: ['baseline', 'fixed14']
District GDP rows: 3705
District self-distance rows: 247


## 3) Core economic building blocks

This section contains the key analytical functions:
- population-weighted district-to-district distance aggregation,
- partition-border dummy by partition difference,
- domestic MA contribution,
- foreign MA contribution.


In [3]:
def compute_district_distance_matrix(dm: pd.DataFrame, point_pop: pd.DataFrame) -> pd.DataFrame:
    # Collapse point-level distances into district-level weighted means.
    pp = point_pop[["point_id", "District", "pop"]].copy()

    x = dm.merge(pp, left_on="origin_id", right_on="point_id", how="left").rename(
        columns={"District": "origin_district", "pop": "origin_pop"}
    )
    x = x.merge(pp, left_on="dest_id", right_on="point_id", how="left", suffixes=("", "_dest")).rename(
        columns={"District": "dest_district", "pop": "dest_pop"}
    )

    x = x.dropna(subset=["origin_district", "dest_district"]).copy()
    x = x[x["origin_district"] != x["dest_district"]].copy()

    x["distance_value"] = pd.to_numeric(x["distance_value"], errors="coerce")
    x = x[x["distance_value"].notna()].copy()

    x["w"] = x["origin_pop"].fillna(0.0) * x["dest_pop"].fillna(0.0)
    x = x[x["w"] > 0].copy()
    x["wd"] = x["w"] * x["distance_value"]

    g = x.groupby(["origin_district", "dest_district"], as_index=False).agg(w_sum=("w", "sum"), wd_sum=("wd", "sum"))
    g["distance_value"] = g["wd_sum"] / g["w_sum"]
    return g[["origin_district", "dest_district", "distance_value"]].copy()


def compute_district_to_border_distance(dm: pd.DataFrame, point_pop: pd.DataFrame) -> pd.DataFrame:
    # For each district and border crossing, compute weighted mean distance from district points.
    pp = point_pop[["point_id", "District", "pop"]].copy()

    x = dm.merge(pp, left_on="origin_id", right_on="point_id", how="left")
    x = x.rename(columns={"District": "origin_district", "pop": "origin_pop"})

    x = x[x["dest_id"].astype(str).str.startswith("Border_Crossing:")].copy()
    x = x.dropna(subset=["origin_district"]).copy()

    x["distance_value"] = pd.to_numeric(x["distance_value"], errors="coerce")
    x["origin_pop"] = pd.to_numeric(x["origin_pop"], errors="coerce").fillna(0.0)
    x = x[x["distance_value"].notna() & (x["origin_pop"] > 0)].copy()
    x["wd"] = x["origin_pop"] * x["distance_value"]

    g = x.groupby(["origin_district", "dest_id"], as_index=False).agg(w_sum=("origin_pop", "sum"), wd_sum=("wd", "sum"))
    g["distance_to_border"] = g["wd_sum"] / g["w_sum"]
    return g[["origin_district", "dest_id", "distance_to_border"]].copy()


def compute_partition_pair_dummy(district_distances: pd.DataFrame, partition_lookup: pd.DataFrame) -> pd.Series:
    # Binary dummy: 1 if origin and destination belonged to different historic partitions.
    part_map = dict(zip(partition_lookup["District"], partition_lookup["partition_label"]))
    o = district_distances["origin_district"].map(part_map).fillna("UNKNOWN")
    d = district_distances["dest_district"].map(part_map).fillna("UNKNOWN")
    return (o != d).astype(int)


def compute_domestic_ma(
    district_distances: pd.DataFrame,
    district_mass: pd.DataFrame,
    partition_lookup: pd.DataFrame,
    coeff_part_border_year: float,
    district_self_distance: pd.DataFrame,
    scenario: str,
) -> pd.DataFrame:
    # Within-Poland MA component with self term based on district area.
    mass_map = dict(zip(district_mass["District"], district_mass["mass"]))

    # Inter-district term (j != i)
    x = district_distances.copy()
    x["dest_mass"] = x["dest_district"].map(mass_map).fillna(0.0)
    x["part_dummy"] = compute_partition_pair_dummy(x, partition_lookup)

    x["contrib_domestic"] = x["dest_mass"] * np.exp(
        COEFF_DISTANCE_DOMESTIC * np.log(x["distance_value"] + EPS_DISTANCE)
        + coeff_part_border_year * x["part_dummy"]
    )

    inter = x.groupby("origin_district", as_index=False)["contrib_domestic"].sum().rename(
        columns={"origin_district": "District", "contrib_domestic": "ma_domestic_inter"}
    )

    # Self term (j == i) using equivalent-circle internal distance.
    self_df = district_mass[["District", "mass"]].merge(district_self_distance, on="District", how="left")

    if scenario == "baseline":
        if COMPUTE_IN_MILES:
            self_df["self_distance"] = self_df["self_distance_km"] * KM_TO_MILES
        else:
            self_df["self_distance"] = self_df["self_distance_km"]
    else:
        # fixed14 uses minutes in distance matrix; convert self km to minutes.
        self_df["self_distance"] = self_df["self_distance_km"] * (60.0 / FIXED14_EXTERNAL_SPEED_KMPH)

    self_df["ma_domestic_self"] = self_df["mass"] * np.exp(
        COEFF_DISTANCE_DOMESTIC * np.log(self_df["self_distance"] + EPS_DISTANCE)
    )

    out = district_mass[["District"]].copy()
    out = out.merge(inter, on="District", how="left")
    out = out.merge(self_df[["District", "ma_domestic_self"]], on="District", how="left")

    out["ma_domestic_inter"] = out["ma_domestic_inter"].fillna(0.0)
    out["ma_domestic_self"] = out["ma_domestic_self"].fillna(0.0)
    out["ma_domestic"] = out["ma_domestic_inter"] + out["ma_domestic_self"]

    return out[["District", "ma_domestic", "ma_domestic_inter", "ma_domestic_self"]].copy()


def compute_foreign_ma(
    year: int,
    scenario: str,
    district_to_border: pd.DataFrame,
    border_connections: pd.DataFrame,
    foreign_gdp: pd.DataFrame,
) -> pd.DataFrame:
    # Foreign MA component through mapped border crossings and external connector lengths.
    bd = district_to_border.copy()
    bd["border_norm"] = bd["dest_id"].map(border_id_norm)

    conn_match = match_border_connections_to_matrix_borders(bd, border_connections)
    if conn_match.empty:
        return pd.DataFrame(columns=["District", "ma_foreign"])

    merged = bd.merge(conn_match, left_on="border_norm", right_on="matched_border_norm", how="inner")
    if merged.empty:
        return pd.DataFrame(columns=["District", "ma_foreign"])

    merged["region_norm"] = merged["country_or_province"].map(normalize_text)

    if scenario == "fixed14":
        connector_length = merged["length_km"] * (KM_TO_MILES if COMPUTE_IN_MILES else 1.0)
        speed = FIXED14_EXTERNAL_SPEED_MPH if COMPUTE_IN_MILES else FIXED14_EXTERNAL_SPEED_KMPH
        connector_distance = connector_length * (60.0 / speed)
    else:
        connector_distance = merged["length_km"] * (KM_TO_MILES if COMPUTE_IN_MILES else 1.0)

    merged["distance_to_foreign"] = merged["distance_to_border"] + connector_distance

    # If multiple routes map to one region, keep the shortest effective route.
    merged = merged.groupby(["origin_district", "region_norm"], as_index=False)["distance_to_foreign"].min()

    gdp_y = foreign_gdp[foreign_gdp["year"] == year][["region_norm", "gdp"]].copy()
    x = merged.merge(gdp_y, on="region_norm", how="inner")

    x["contrib_foreign"] = x["gdp"] * np.exp(COEFF_DISTANCE_FOREIGN * np.log(x["distance_to_foreign"] + EPS_DISTANCE))

    out = x.groupby("origin_district", as_index=False)["contrib_foreign"].sum().rename(
        columns={"origin_district": "District", "contrib_foreign": "ma_foreign"}
    )
    return out


## 4) Scenario runner

For each available year in a scenario:
1. Load distance matrix,
2. Build district masses from rural + city populations,
3. Compute domestic MA and foreign MA,
4. Store total + decomposition.


In [4]:
def run_scenario(scenario: str) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    years = list_available_years(DIST_DIR, scenario, YEARS_TARGET)
    print(f"Scenario={scenario}; years found: {years}")
    if not years:
        empty_ma = pd.DataFrame(columns=["District", "year", "scenario", "ma_total", "ma_domestic", "ma_foreign"])
        empty_dist = pd.DataFrame(columns=["scenario", "year", "origin_district", "dest_district", "distance_value"])
        empty_border = pd.DataFrame(columns=["scenario", "year", "origin_district", "dest_id", "distance_to_border"])
        return empty_ma, empty_dist, empty_border

    rows_ma = []
    rows_dist = []
    rows_border = []

    for year in years:
        print("  year", year)

        dm = load_distance_matrix_for_scenario(
            DIST_DIR,
            year,
            scenario,
            compute_in_miles=COMPUTE_IN_MILES,
        )
        point_pop, _ = build_point_population(year, city_keep, rural_keep)

        district_mass = district_gdp_long[district_gdp_long["year"] == year][["District", "district_gdp"]].copy()
        district_mass = district_mass.rename(columns={"district_gdp": "mass"})

        district_distances = compute_district_distance_matrix(dm, point_pop)
        district_distances = district_distances.copy()
        district_distances["scenario"] = scenario
        district_distances["year"] = year
        rows_dist.append(
            district_distances[["scenario", "year", "origin_district", "dest_district", "distance_value"]]
        )

        district_to_border = compute_district_to_border_distance(dm, point_pop)
        district_to_border = district_to_border.copy()
        district_to_border["scenario"] = scenario
        district_to_border["year"] = year
        rows_border.append(
            district_to_border[["scenario", "year", "origin_district", "dest_id", "distance_to_border"]]
        )

        coeff_part = partition_coeff.get(year)
        if coeff_part is None:
            raise RuntimeError(f"Missing partition coefficient for {year}")

        dom = compute_domestic_ma(
            district_distances,
            district_mass,
            partition_df,
            coeff_part,
            district_self_distance,
            scenario,
        )
        foreign = compute_foreign_ma(year, scenario, district_to_border, border_conn, foreign_gdp_long)

        out = district_mass[["District"]].copy()
        out["year"] = year
        out["scenario"] = scenario
        out = out.merge(dom, on="District", how="left")
        out = out.merge(foreign, on="District", how="left")

        out["ma_domestic"] = out["ma_domestic"].fillna(0.0)
        out["ma_foreign"] = out["ma_foreign"].fillna(0.0)
        out["ma_total"] = out["ma_domestic"] + out["ma_foreign"]

        rows_ma.append(out[["District", "year", "scenario", "ma_total", "ma_domestic", "ma_foreign"]])

    ma_df = pd.concat(rows_ma, ignore_index=True)
    dist_df = pd.concat(rows_dist, ignore_index=True)
    border_df = pd.concat(rows_border, ignore_index=True)
    return ma_df, dist_df, border_df


scenario_results = [run_scenario(sc) for sc in SCENARIOS]
ma_all = pd.concat([r[0] for r in scenario_results], ignore_index=True)
district_distances_all = pd.concat([r[1] for r in scenario_results], ignore_index=True)
district_to_border_all = pd.concat([r[2] for r in scenario_results], ignore_index=True)

print("Rows computed (ma_all):", len(ma_all))
print("Rows computed (district_distances_all):", len(district_distances_all))
print("Rows computed (district_to_border_all):", len(district_to_border_all))
ma_all.head()


Scenario=baseline; years found: [1924, 1925, 1926, 1927, 1928, 1929, 1930, 1931, 1932, 1933, 1934, 1935, 1936, 1937, 1938]
  year 1924


  year 1925
  year 1926
  year 1927
  year 1928
  year 1929
  year 1930
  year 1931
  year 1932
  year 1933
  year 1934
  year 1935
  year 1936
  year 1937
  year 1938
Scenario=fixed14; years found: [1924, 1925, 1926, 1927, 1928, 1929, 1930, 1931, 1932, 1933, 1934, 1935, 1936, 1937, 1938]
  year 1924
  year 1925
  year 1926
  year 1927
  year 1928
  year 1929
  year 1930
  year 1931
  year 1932
  year 1933
  year 1934
  year 1935
  year 1936
  year 1937
  year 1938
Rows computed (ma_all): 7410
Rows computed (district_distances_all): 1822860
Rows computed (district_to_border_all): 37050


,District,year,scenario,ma_total,ma_domestic,ma_foreign
0,AUGUSTOWSKI,1924,baseline,6.181654e+08,53806.285334,6.181116e+08
1,BARANOWICKI,1924,baseline,5.940234e+08,44789.030471,5.939786e+08
2,BIALSKI (BIAŁA KRAKOWSKA),1924,baseline,8.790968e+08,961405.345355,8.781354e+08
3,BIALSKI (BIAŁA PODLASKA),1924,baseline,7.156088e+08,111935.321183,7.154968e+08
4,BIAŁOSTOCKI,1924,baseline,6.863698e+08,176533.012245,6.861933e+08


## 5) Quick validation checks

These checks are intentionally simple and transparent for auditability.


In [5]:
if ma_all.empty:
    print("No results computed yet (missing matrices in target years).")
else:
    coverage = ma_all.groupby(["scenario", "year"], as_index=False).agg(
        districts=("District", "nunique"),
        ma_total_min=("ma_total", "min"),
        ma_total_median=("ma_total", "median"),
        ma_total_max=("ma_total", "max"),
    )
    display(coverage)

    check_add = (ma_all["ma_total"] - (ma_all["ma_domestic"] + ma_all["ma_foreign"]))
    print("Max decomposition error:", float(np.abs(check_add).max()))


,scenario,year,districts,ma_total_min,ma_total_median,ma_total_max
0,baseline,1924,247,5.356010e+08,7.457583e+08,1.208347e+09
1,baseline,1925,247,5.937762e+08,8.266980e+08,1.340305e+09
2,baseline,1926,247,6.090261e+08,8.559442e+08,1.419017e+09
3,baseline,1927,247,6.715510e+08,9.435164e+08,1.563989e+09
4,baseline,1928,247,7.048499e+08,9.897672e+08,1.640235e+09
5,baseline,1929,247,6.981630e+08,9.808525e+08,1.625948e+09
6,baseline,1930,247,6.894328e+08,9.686149e+08,1.605696e+09
7,baseline,1931,247,6.369806e+08,8.950736e+08,1.483686e+09
8,baseline,1932,247,5.894325e+08,8.281053e+08,1.373695e+09
9,baseline,1933,247,6.260824e+08,8.800410e+08,1.457910e+09


Max decomposition error: 0.0


In [6]:
TARGET_DISTANCE_PAIRS = [
    ("M. ST. WARSZAWA", "KRAKÓW (MIASTO)"),
    ("M. ST. WARSZAWA", "WILNO (MIASTO)"),
    ("M. ST. WARSZAWA", "LWÓW (MIASTO)"),
    ("M. ST. WARSZAWA", "BIAŁOSTOCKI"),
    ("M. ST. WARSZAWA", "MORSKI"),
    ("BIAŁOSTOCKI", "WILNO (MIASTO)"),
    ("KRAKÓW (MIASTO)", "KIELECKI"),
    ("BIAŁOSTOCKI", "LWÓW (MIASTO)"),
    ("MORSKI", "BIELSKO"),
    ("POZNAŃ (MIASTO)", "BIELSKO"),
]


distance_check = collect_target_pair_distances(
    ma_all=ma_all,
    district_distances_all=district_distances_all,
    compute_in_miles=COMPUTE_IN_MILES,
    target_distance_pairs=TARGET_DISTANCE_PAIRS,
)

display(distance_check.loc[(distance_check["scenario"]=="baseline") & (distance_check["year"]==1938)].sort_values(["scenario", "year", "origin_district", "dest_district"]))


,scenario,year,origin_district,dest_district,distance_value,distance_km,distance_miles,distance_minutes
147,baseline,1938,BIAŁOSTOCKI,LWÓW (MIASTO),262.132875,421.862100,262.132875,NaN
145,baseline,1938,BIAŁOSTOCKI,WILNO (MIASTO),146.346060,235.521226,146.346060,NaN
146,baseline,1938,KRAKÓW (MIASTO),KIELECKI,80.667420,129.821669,80.667420,NaN
143,baseline,1938,M. ST. WARSZAWA,BIAŁOSTOCKI,116.826181,188.013572,116.826181,NaN
140,baseline,1938,M. ST. WARSZAWA,KRAKÓW (MIASTO),182.337337,293.443590,182.337337,NaN
142,baseline,1938,M. ST. WARSZAWA,LWÓW (MIASTO),249.426731,401.413537,249.426731,NaN
144,baseline,1938,M. ST. WARSZAWA,MORSKI,257.721046,414.761948,257.721046,NaN
141,baseline,1938,M. ST. WARSZAWA,WILNO (MIASTO),253.193253,407.475169,253.193253,NaN
148,baseline,1938,MORSKI,BIELSKO,362.459657,583.322455,362.459657,NaN
149,baseline,1938,POZNAŃ (MIASTO),BIELSKO,233.297278,375.455691,233.297278,NaN


In [7]:
WARSAW_FOREIGN_CITIES = ["Berlin", "Praha", "Kyiv"]

warsaw_foreign_distance_check = collect_foreign_city_route_distances(
    ma_all=ma_all[ma_all["scenario"] == "baseline"],
    district_to_border_all=district_to_border_all,
    border_connections=border_conn,
    origin_district="M. ST. WARSZAWA",
    foreign_cities=WARSAW_FOREIGN_CITIES,
    compute_in_miles=COMPUTE_IN_MILES,
)

if warsaw_foreign_distance_check.empty:
    print("No matched Warsaw->(Berlin/Praha/Kyiv) routes found for current data.")
else:
    display(warsaw_foreign_distance_check.sort_values(["scenario", "year", "foreign_city"]))


,scenario,year,origin_district,foreign_city,border_crossing,district_to_border_km,district_to_border_miles,border_to_city_km,border_to_city_miles,total_km,total_miles
0,baseline,1924,M. ST. WARSZAWA,Berlin,Zbąszyń,385.573582,239.584242,174.522899,108.443468,560.096481,348.027710
1,baseline,1925,M. ST. WARSZAWA,Berlin,Zbąszyń,385.573582,239.584242,174.522899,108.443468,560.096481,348.027710
2,baseline,1926,M. ST. WARSZAWA,Berlin,Zbąszyń,385.573581,239.584242,174.522899,108.443468,560.096480,348.027710
3,baseline,1927,M. ST. WARSZAWA,Berlin,Zbąszyń,385.573581,239.584242,174.522899,108.443468,560.096480,348.027710
4,baseline,1928,M. ST. WARSZAWA,Berlin,Zbąszyń,385.573581,239.584241,174.522899,108.443468,560.096480,348.027710
5,baseline,1929,M. ST. WARSZAWA,Berlin,Zbąszyń,385.573581,239.584241,174.522899,108.443468,560.096480,348.027710
6,baseline,1930,M. ST. WARSZAWA,Berlin,Zbąszyń,385.573580,239.584241,174.522899,108.443468,560.096479,348.027709
7,baseline,1931,M. ST. WARSZAWA,Berlin,Zbąszyń,385.573580,239.584241,174.522899,108.443468,560.096479,348.027709
8,baseline,1932,M. ST. WARSZAWA,Berlin,Zbąszyń,385.573580,239.584241,174.522899,108.443468,560.096479,348.027709
9,baseline,1933,M. ST. WARSZAWA,Berlin,Zbąszyń,385.949863,239.818052,174.522899,108.443468,560.472762,348.261520


In [8]:
# Nearest-3 domestic contributors by district (name, distance, contribution)

# Precompute district GDP mass by year.
mass_by_year = {}
for y in sorted(ma_all["year"].unique()):
    g = district_gdp_long[district_gdp_long["year"] == int(y)][["District", "district_gdp"]].copy()
    g = g.rename(columns={"district_gdp": "mass"})
    mass_by_year[int(y)] = g

# Partition labels map.
part_map = dict(zip(partition_df["District"], partition_df["partition_label"]))

x = district_distances_all.copy()
x["dest_mass"] = np.nan
for y in sorted(x["year"].unique()):
    m = mass_by_year[int(y)]
    m_map = dict(zip(m["District"], m["mass"]))
    mask = x["year"] == y
    x.loc[mask, "dest_mass"] = x.loc[mask, "dest_district"].map(m_map)

x["origin_part"] = x["origin_district"].map(part_map).fillna("UNKNOWN")
x["dest_part"] = x["dest_district"].map(part_map).fillna("UNKNOWN")
x["part_dummy"] = (x["origin_part"] != x["dest_part"]).astype(int)
x["gamma_t"] = x["year"].map(partition_coeff)

x["domestic_contribution"] = x["dest_mass"] * np.exp(
    COEFF_DISTANCE_DOMESTIC * np.log(x["distance_value"] + EPS_DISTANCE)
    + x["gamma_t"] * x["part_dummy"]
)

# Keep 3 nearest destination districts for each (scenario, year, origin_district).
x = x.sort_values(["scenario", "year", "origin_district", "distance_value"]).copy()
x["rank_nearest"] = x.groupby(["scenario", "year", "origin_district"]).cumcount() + 1
nearest3_domestic = x[x["rank_nearest"] <= 3].copy()

# Add distance columns in both units.
nearest3_domestic["distance_minutes"] = np.where(
    nearest3_domestic["scenario"] == "fixed14",
    nearest3_domestic["distance_value"],
    np.nan,
)

if COMPUTE_IN_MILES:
    nearest3_domestic["distance_miles"] = np.where(
        nearest3_domestic["scenario"] == "baseline",
        nearest3_domestic["distance_value"],
        np.nan,
    )
    nearest3_domestic["distance_km"] = nearest3_domestic["distance_miles"] / 0.621371
else:
    nearest3_domestic["distance_km"] = np.where(
        nearest3_domestic["scenario"] == "baseline",
        nearest3_domestic["distance_value"],
        np.nan,
    )
    nearest3_domestic["distance_miles"] = nearest3_domestic["distance_km"] * 0.621371

nearest3_domestic = nearest3_domestic[
    [
        "scenario",
        "year",
        "origin_district",
        "rank_nearest",
        "dest_district",
        "distance_km",
        "distance_miles",
        "distance_minutes",
        "domestic_contribution",
    ]
].copy()

# Show a compact preview (all rows can be large).
display(nearest3_domestic.sort_values(["scenario", "year", "origin_district", "rank_nearest"]).head(120))

# Export nearest-3 diagnostics to dedicated checks folder.
checks_dir = OUT_DIR / "checks"
checks_dir.mkdir(parents=True, exist_ok=True)

nearest3_csv = checks_dir / "nearest3_domestic.csv"
nearest3_domestic.to_csv(nearest3_csv, index=False)
print("Wrote:", nearest3_csv)

nearest3_xlsx = checks_dir / "nearest3_domestic.xlsx"
try:
    nearest3_domestic.to_excel(nearest3_xlsx, index=False)
    print("Wrote:", nearest3_xlsx)
except Exception as exc:
    print(f"Could not write xlsx: {exc}")



,scenario,year,origin_district,rank_nearest,dest_district,distance_km,distance_miles,distance_minutes,domestic_contribution
182,baseline,1924,AUGUSTOWSKI,1,SUWALSKI,49.201213,30.572207,NaN,4831.142848
46,baseline,1924,AUGUSTOWSKI,2,GRODZIEŃSKI,98.919709,61.465838,NaN,1197.451823
175,baseline,1924,AUGUSTOWSKI,3,SOKÓLSKI,117.002972,72.702254,NaN,302.301903
363,baseline,1924,BARANOWICKI,1,NIEŚWIESKI,94.804322,58.908656,NaN,807.097634
434,baseline,1924,BARANOWICKI,2,SŁONIMSKI,101.697580,63.191927,NaN,544.427414
...,...,...,...,...,...,...,...,...,...
9360,baseline,1924,DUBIEŃSKI,2,BRODZKI,63.060603,39.184030,NaN,241.473023
9510,baseline,1924,DUBIEŃSKI,3,RÓWIEŃSKI,84.296715,52.379534,NaN,1913.745031
9708,baseline,1924,DZIAŁDOWSKI,1,MŁAWSKI,41.784647,25.963768,NaN,1414.462180
9605,baseline,1924,DZIAŁDOWSKI,2,BRODNICKI,63.987412,39.759922,NaN,1827.347353


Wrote: examples\interwar_poland\market_access\outputs\market_access\checks\nearest3_domestic.csv
Wrote: examples\interwar_poland\market_access\outputs\market_access\checks\nearest3_domestic.xlsx


## 6) Export tables

Exports are scenario-specific and include:
- annual total MA,
- annual within-country MA,
- annual foreign-only MA.


In [9]:
for sc in SCENARIOS:
    save_tables(ma_all, OUT_DIR, sc)

# Export district-to-district distance matrices used in MA computations.
dist_out_dir = OUT_DIR / "distance_matrices"
dist_out_dir.mkdir(parents=True, exist_ok=True)

# Combined long table (all scenarios/years)
all_dist_path = dist_out_dir / "district_distance_matrices_all.csv"
district_distances_all.to_csv(all_dist_path, index=False)
print("Wrote:", all_dist_path)

# Scenario/year-specific files
for (sc, yr), grp in district_distances_all.groupby(["scenario", "year"], as_index=False):
    p = dist_out_dir / f"district_distance_matrix_{sc}_{int(yr)}.csv"
    grp.to_csv(p, index=False)
    print("Wrote:", p)

# Export log changes (1938 vs 1924) for MA variants.
log_change_rows = []
for sc in SCENARIOS:
    d = ma_all[ma_all["scenario"] == sc].copy()
    if d.empty:
        continue
    if 1924 not in set(d["year"]) or 1938 not in set(d["year"]):
        continue

    a = d[d["year"] == 1924][["District", "ma_total", "ma_domestic", "ma_foreign"]].rename(
        columns={
            "ma_total": "ma_total_1924",
            "ma_domestic": "ma_domestic_1924",
            "ma_foreign": "ma_foreign_1924",
        }
    )
    b = d[d["year"] == 1938][["District", "ma_total", "ma_domestic", "ma_foreign"]].rename(
        columns={
            "ma_total": "ma_total_1938",
            "ma_domestic": "ma_domestic_1938",
            "ma_foreign": "ma_foreign_1938",
        }
    )
    m = a.merge(b, on="District", how="inner")
    eps = 1e-12
    m["scenario"] = sc
    m["log_change_ma_total"] = np.log(m["ma_total_1938"] + eps) - np.log(m["ma_total_1924"] + eps)
    m["log_change_ma_domestic"] = np.log(m["ma_domestic_1938"] + eps) - np.log(m["ma_domestic_1924"] + eps)
    m["log_change_ma_foreign"] = np.log(m["ma_foreign_1938"] + eps) - np.log(m["ma_foreign_1924"] + eps)
    log_change_rows.append(m[["scenario", "District", "log_change_ma_total", "log_change_ma_domestic", "log_change_ma_foreign"]])

if log_change_rows:
    ma_log_change = pd.concat(log_change_rows, ignore_index=True)
    log_change_path = OUT_DIR / "checks" / "ma_log_change_1938_vs_1924.csv"
    log_change_path.parent.mkdir(parents=True, exist_ok=True)
    ma_log_change.to_csv(log_change_path, index=False)
    print("Wrote:", log_change_path)



Wrote: examples\interwar_poland\market_access\outputs\market_access\baseline\market_access_annual_baseline.csv
Wrote: examples\interwar_poland\market_access\outputs\market_access\baseline\market_access_annual_baseline.xlsx
Wrote: examples\interwar_poland\market_access\outputs\market_access\fixed14\market_access_annual_fixed14.csv
Wrote: examples\interwar_poland\market_access\outputs\market_access\fixed14\market_access_annual_fixed14.xlsx
Wrote: examples\interwar_poland\market_access\outputs\market_access\distance_matrices\district_distance_matrices_all.csv
Wrote: examples\interwar_poland\market_access\outputs\market_access\distance_matrices\district_distance_matrix_baseline_1924.csv
Wrote: examples\interwar_poland\market_access\outputs\market_access\distance_matrices\district_distance_matrix_baseline_1925.csv
Wrote: examples\interwar_poland\market_access\outputs\market_access\distance_matrices\district_distance_matrix_baseline_1926.csv
Wrote: examples\interwar_poland\market_access\outp

## 7) Maps

Generated for each scenario:
- annual maps (1924-1938 as available) for total / domestic / foreign MA,
- change maps for 1938 vs 1924 for all three measures.


In [10]:
# Log variants for additional annual maps.
LOG_EPS = 1e-12
ma_all["ma_total_log"] = np.log(ma_all["ma_total"].astype(float) + LOG_EPS)
ma_all["ma_domestic_log"] = np.log(ma_all["ma_domestic"].astype(float) + LOG_EPS)
ma_all["ma_foreign_log"] = np.log(ma_all["ma_foreign"].astype(float) + LOG_EPS)

for sc in SCENARIOS:
    save_annual_maps(ma_all, sc, "ma_total", "OrRd", PLOTS_DIR, adm_history_plotter, ADM_STATE_DATE, d_city_mapping)
    save_annual_maps(ma_all, sc, "ma_domestic", "YlGnBu", PLOTS_DIR, adm_history_plotter, ADM_STATE_DATE, d_city_mapping)
    save_annual_maps(ma_all, sc, "ma_foreign", "PuRd", PLOTS_DIR, adm_history_plotter, ADM_STATE_DATE, d_city_mapping)

    # New log-scale level maps
    save_annual_maps(ma_all, sc, "ma_total_log", "viridis", PLOTS_DIR, adm_history_plotter, ADM_STATE_DATE, d_city_mapping)
    save_annual_maps(ma_all, sc, "ma_domestic_log", "cividis", PLOTS_DIR, adm_history_plotter, ADM_STATE_DATE, d_city_mapping)
    save_annual_maps(ma_all, sc, "ma_foreign_log", "magma", PLOTS_DIR, adm_history_plotter, ADM_STATE_DATE, d_city_mapping)

    save_change_map_1938_vs_1924(ma_all, sc, "ma_total", "RdBu", PLOTS_DIR, adm_history_plotter, ADM_STATE_DATE, d_city_mapping)
    save_change_map_1938_vs_1924(ma_all, sc, "ma_domestic", "RdBu", PLOTS_DIR, adm_history_plotter, ADM_STATE_DATE, d_city_mapping)
    save_change_map_1938_vs_1924(ma_all, sc, "ma_foreign", "RdBu", PLOTS_DIR, adm_history_plotter, ADM_STATE_DATE, d_city_mapping)


    # New log-change maps (difference in log MA: 1938 vs 1924)
    save_change_map_1938_vs_1924(ma_all, sc, "ma_total_log", "RdBu", PLOTS_DIR, adm_history_plotter, ADM_STATE_DATE, d_city_mapping)
    save_change_map_1938_vs_1924(ma_all, sc, "ma_domestic_log", "RdBu", PLOTS_DIR, adm_history_plotter, ADM_STATE_DATE, d_city_mapping)
    save_change_map_1938_vs_1924(ma_all, sc, "ma_foreign_log", "RdBu", PLOTS_DIR, adm_history_plotter, ADM_STATE_DATE, d_city_mapping)

print("Finished maps (including log maps).")


Finished maps (including log maps).


## Notes for reviewers

- Partition-border effect is a **binary partition difference** indicator.
- Gdańsk treatment follows domestic distance coefficient.
- `COMPUTE_IN_MILES=True` converts km-based distances to miles (baseline + foreign connectors).
